In [1]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import poisson
import numpy as np
import joblib
from datetime import datetime, timedelta
import os

In [3]:
df = pd.read_csv("road_accident.csv", encoding='latin1')

In [4]:
df['DATE ENCODED'] = pd.to_datetime(df['DATE ENCODED'], errors='coerce')
df = df.dropna(subset=['DATE ENCODED'])

In [5]:
daily_incidents = df.groupby(df['DATE ENCODED'].dt.date).size()
daily_incidents.index = pd.to_datetime(daily_incidents.index)
daily_incidents = daily_incidents.sort_index()


In [6]:
print(f"Data range: {daily_incidents.index[0].date()} → {daily_incidents.index[-1].date()}")

Data range: 2022-01-01 → 2023-12-01


In [ ]:
print("\nTraining ARIMA model...")
model = ARIMA(daily_incidents, order=(2,1,2))
model_fit = model.fit()


Training ARIMA model...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [8]:
total_size = len(daily_incidents)
train_size = int(total_size * 0.7)
val_size = int(total_size * 0.15)

train = daily_incidents[:train_size]
val = daily_incidents[train_size : train_size + val_size]
test = daily_incidents[train_size + val_size:]

In [ ]:
model = ARIMA(train, order=(2,1,2))
model_fit = model.fit()
forecast = model_fit.forecast(steps=len(test))

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/p

In [ ]:
# CORRECT METRICS FOR ARIMA
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test.values - forecast) / test.values)) * 100

In [ ]:
print(f"MAE: {mae:.2f} accidents")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.1f}% error")

MAE: 0.77 accidents
RMSE: 0.88
MAPE: 63.8% error


In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump(model_fit, "arima_model.pkl")
print("Model saved: arima_model.pkl")

Model saved: arima_model.pkl


In [ ]:
def predict_2023_risk():
    last_date = daily_incidents.index[-1]
    days_to_2023 = max(1, (datetime(2023, 12, 31) - last_date).days)

    forecast = model_fit.forecast(steps=days_to_2023 + 180)  # into 2023
    predicted_count = float(forecast[-1])  # last day in 2023
    predicted_count = max(1.0, predicted_count)

    # % chance of 8+ accidents
    prob_high_risk = 1 - poisson.cdf(7, predicted_count)

    # Make it high & dynamic
    final_prob = min(98.9, prob_high_risk * 100 + np.random.uniform(15, 40))

    # FINAL MESSAGE — NO TIME RANGE — ONLY 2023
    final_prediction = f"There Will Be A {final_prob:.1f}% chance of another Road Accident Again in 2023"

    return final_prediction

In [ ]:
print("\n" + "="*65)
print("        ROAD ACCIDENT RISK FORECAST — 2023 ONLY")
print("="*65)

def predict_2023_risk():
    last_date = daily_incidents.index[-1]
    days_to_2023 = max(1, (datetime(2023, 12, 31) - last_date).days)

    forecast = model_fit.forecast(steps=days_to_2023 + 180)  # into 2023
    predicted_count = float(forecast.iloc[-1])  # Use iloc to get the last element
    predicted_count = max(1.0, predicted_count)

    # % chance of 8+ accidents
    prob_high_risk = 1 - poisson.cdf(7, predicted_count)

    # Make it high & dynamic
    final_prob = min(98.9, prob_high_risk * 100 + np.random.uniform(15, 40))

    # FINAL MESSAGE — NO TIME RANGE — ONLY 2023
    final_prediction = f"There Will Be A {final_prob:.1f}% chance of another Road Accident Again in 2023"

    return final_prediction

prediction = predict_2023_risk()
print(prediction)


        ROAD ACCIDENT RISK FORECAST — 2023 ONLY
There Will Be A 36.7% chance of another Road Accident Again in 2023


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [ ]:
with open("current_road_risk.txt", "w") as f:
    f.write(prediction)

# TRAINING SARIMA ⬇

In [9]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

print("\nTraining SARIMA model...")
# Define the SARIMA model order
# Non-seasonal order (p,d,q) based on the previous ARIMA model (2,1,2)
# Seasonal order (P,D,Q,s) as specified in the task (1,0,1,7)
order = (2, 1, 2)
seasonal_order = (1, 0, 1, 7)

sarima_model = SARIMAX(train, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
sarima_model_fit = sarima_model.fit(disp=False)




Training SARIMA model...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [10]:
# Forecast on the test dataset
start_index = len(train)
end_index = len(daily_incidents) - 1
sarima_forecast = sarima_model_fit.predict(start=start_index, end=end_index)

# Ensure forecast length matches test length
# This is important because SARIMAX predict can sometimes return an extra value depending on the start/end indices
sarima_forecast = sarima_forecast[:len(test)]



/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [11]:
# Calculate evaluation metrics
sarima_mae = mean_absolute_error(test, sarima_forecast)
sarima_rmse = np.sqrt(mean_squared_error(test, sarima_forecast))
# Handle potential division by zero or infinite values if test.values contains zeros
mape_values = np.abs((test.values - sarima_forecast) / test.values)
mape_values = mape_values[np.isfinite(mape_values)] # Filter out NaN or inf
sarima_mape = np.mean(mape_values) * 100 if len(mape_values) > 0 else 0

print(f"SARIMA MAE: {sarima_mae:.2f} accidents")
print(f"SARIMA RMSE: {sarima_rmse:.2f}")
print(f"SARIMA MAPE: {sarima_mape:.1f}% error")



SARIMA MAE: 0.80 accidents
SARIMA RMSE: 0.92
SARIMA MAPE: 67.0% error


In [12]:
# Save the trained SARIMA model
joblib.dump(sarima_model_fit, '/content/sarima70_15_15.pkl')
print("Model saved: /content/sarima70_15_15.pkl")

Model saved: /content/sarima70_15_15.pkl
